# Phase 3 — Nigerian Pidgin (pcm) fine-tuning, adapted scope

Gated notebook — run stages **in order**, each one is a checkpoint:

| Stage | What | GPU cost |
|---|---|---|
| 0 | Hard license gates (PRD §8) | none |
| 1 | Verify the VoxCPM2 fine-tune API + manifest schema | none |
| 2 | Download Common Voice pcm (CC0) | none |
| 3 | Build speaker-disjoint manifests | none |
| 4 | Feasibility micro-run (~50 steps) | minutes |
| 5 | Full run — gated on Stage 4 results | hours |

Never improvise around a failure — stop and report it (task constraint).

In [ ]:
# STAGE 0 — HARD GATES. Proceeds only if all three flags are true.
import os, sys, subprocess
os.chdir('/content')
BASE = '/content/naturalvoice-ai'
if not os.path.isdir(BASE):
    subprocess.run(['git', 'clone', 'https://github.com/Etzkennyboi/native-voice.git', BASE], check=True)
os.chdir(BASE)
if BASE not in sys.path:
    sys.path.insert(0, BASE)

import yaml
cfg = yaml.safe_load(open('configs/pidgin_baseline.yaml', encoding='utf-8'))
lic = cfg['license']
print('license_cleared            :', lic['license_cleared'])
print('consent_review_done        :', lic['consent_review_done'])
print('base_checkpoint_license_ok :', lic['base_checkpoint_license_ok'])
ok = all(lic[k] for k in ('license_cleared', 'consent_review_done', 'base_checkpoint_license_ok'))
print()
print('GATE:', 'PASS - go to Stage 1' if ok else
      'BLOCKED - set the flags ONLY after the licence reviews in docs/pidgin_dataset_sourcing.md (L1-L4)')
assert ok, 'License gates not cleared - training stays blocked (PRD section 8, hard rule).'

In [ ]:
# STAGE 1 — VERIFY the fine-tuning API + manifest schema BEFORE building manifests.
# Zero GPU cost. Never assume the schema matches our baseline format (task constraint).
!pip install -q voxcpm==2.0.3
import voxcpm, pathlib
root = pathlib.Path(voxcpm.__file__).resolve().parent
print('voxcpm version:', getattr(voxcpm, '__version__', '?'))
print('package root  :', root)
hits = []
for p in root.rglob('*.py'):
    t = p.read_text(errors='ignore').lower()
    if any(k in t for k in ('finetune', 'lora_config', 'training', 'manifest', 'jsonl', 'dataset')):
        hits.append(str(p.relative_to(root)))
print('files mentioning finetune/lora/dataset/manifest:')
for h in hits:
    print('  ', h)
print()
print('--- CLI help (look for a train / finetune subcommand) ---')
get_ipython().system('voxcpm --help')
print()
print('NEXT (manual, but mechanical): open the finetune/training module listed above,')
print('find (a) the training entrypoint + its LoRA config fields, (b) the EXACT manifest')
print('keys it reads (audio path? text? speaker?). Then:')
print('  1. update SCHEMA_KEYS in scripts/build_pidgin_manifests.py')
print('  2. set data.split_locked: true in configs/pidgin_baseline.yaml')
print('  3. define FT_ENTRY in Stage 4 cell from the verified entrypoint')
print('      (also set license.base_checkpoint_license_ok after reading the VoxCPM2 license)')

In [ ]:
# STAGE 2 — download Common Voice pcm (primary source, CC0-1.0).
# Gated repo: accept the terms once in the browser on the Data Collective / HF page,
# then paste the token below. The token is NEVER printed or saved to the repo (PRD §10).
from huggingface_hub import login, snapshot_download
import getpass
login(token=getpass.getpass('HF token (input hidden): '))
local = snapshot_download(
    'mozilla-foundation/common_voice_25_0',
    repo_type='dataset',
    allow_patterns=['*pcm*'],
    local_dir='/content/cv_pcm',
)
print('downloaded to:', local)

In [ ]:
# STAGE 3 — build speaker-disjoint manifests. Runs only when the gates in the
# config are true (the script re-checks them and refuses otherwise).
# Adjust the two paths to whatever Stage 1/2 actually produced (CV zips vary by release).
TSV = '/content/cv_pcm/pcm/validated_pcm.tsv'   # <-- check after download
AUDIO = '/content/cv_pcm/pcm/clips'             # <-- check after download
!python scripts/build_pidgin_manifests.py --tsv "$TSV" --audio-dir "$AUDIO"
!wc -l data/manifests/pidgin_train.jsonl data/manifests/pidgin_val.jsonl data/manifests/pidgin_test.jsonl

In [ ]:
# STAGE 4 — feasibility micro-run (PRD section 11: short probe BEFORE the full run).
# ~50 LoRA steps on a tiny manifest slice. Records VRAM peak, wall time, seed.
import json, time, pathlib, torch, yaml
from voxcpm import VoxCPM
from app.tts_engine import _load_model, _sample_rate   # verified adapter (Phase 0/1)

FT_ENTRY = None   # <-- FILL from Stage 1: the verified training call, e.g. ft.train(...)
assert FT_ENTRY is not None, 'Stage 1 must verify the training API first - never assume (task constraint).'

SEED = 3407
torch.manual_seed(SEED)
cfg = yaml.safe_load(open('configs/pidgin_baseline.yaml', encoding='utf-8'))

train_slice = []
with open('data/manifests/pidgin_train.jsonl', encoding='utf-8') as fh:
    for line in fh:
        train_slice.append(json.loads(line))
        if len(train_slice) >= 40:   # ~2-3 minutes of audio
            break

t0 = time.time()
model = _load_model(cfg)                 # pinned snapshot; baseline stays untouched on disk
load_s = time.time() - t0
free, total = torch.cuda.mem_get_info()

t1 = time.time()
# FT_ENTRY(model, train_slice, steps=50, seed=SEED, **cfg['training']['lora'])
raise SystemExit('FILL the FT_ENTRY call from Stage 1 findings, then re-run this cell.')
train_s = time.time() - t1
peak = torch.cuda.max_memory_allocated() / 2**30

record = {
    'stage': 'pidgin_feasibility', 'date': time.strftime('%Y-%m-%d'),
    'seed': SEED, 'steps': 50, 'clips': len(train_slice),
    'model_load_s': round(load_s, 1), 'train_s': round(train_s, 1),
    'vram_peak_gib': round(peak, 2), 'vram_free_gib': round(free / 2**30, 2),
    'gpu_total_gib': round(total / 2**30, 2),
    'text': 'voxcpm 2.0.3, snapshot ' + cfg['base_model']['revision'],
}
out = pathlib.Path('experiments') / ("pidgin-feasibility-" + time.strftime('%Y%m%d-%H%M') + '.json')
out.write_text(json.dumps(record, indent=2), encoding='utf-8')
print(json.dumps(record, indent=2))
print('If VRAM peak is far below the reference 20 GiB, Stage 5 may fit this GPU class;')
print('if it OOMs or explodes - STOP and report, do not tweak around it.')

In [ ]:
# STAGE 5 — full gated run. ONLY after Stage 4 numbers look sane.
import json, time, pathlib, torch, yaml
from app.tts_engine import _load_model

assert FT_ENTRY is not None, 'Run Stage 4 first.'
peaked_ok = True   # <-- flip to True manually after reading the Stage 4 JSON record
assert peaked_ok, 'Stage 5 stays gated until the feasibility record is reviewed.'

SEED = 3407
torch.manual_seed(SEED)
cfg = yaml.safe_load(open('configs/pidgin_baseline.yaml', encoding='utf-8'))
model = _load_model(cfg)

records = [json.loads(l) for l in open('data/manifests/pidgin_train.jsonl', encoding='utf-8')]
out_dir = pathlib.Path('/content/artifacts/voxcpm2-pidgin-v0')
out_dir.mkdir(parents=True, exist_ok=True)

run_log = {
    'checkpoint_id': 'voxcpm2-pidgin-v0',
    'base_revision': cfg['base_model']['revision'],
    'seed': SEED,
    'lora': cfg['training']['lora'],
    'learning_rate': cfg['training']['learning_rate'],
    'epochs': cfg['training']['epochs'],
    'manifest_clips': len(records),
    'gpu': torch.cuda.get_device_name(0),
    'started': time.strftime('%Y-%m-%d %H:%M:%S'),
}
# FT_ENTRY(model, records, epochs=cfg['training']['epochs'], seed=SEED, output_dir=str(out_dir), ...)
raise SystemExit('FILL the FT_ENTRY full-run call from Stage 1, then re-run.')
run_log['finished'] = time.strftime('%Y-%m-%d %H:%M:%S')
pathlib.Path('experiments') / 'pidgin-full-run.json'
(pathlib.Path('experiments') / 'pidgin-full-run.json').write_text(json.dumps(run_log, indent=2), encoding='utf-8')
print('LoRA weights saved under', out_dir)
print('Next: set tuned_weights.lora_weights_path in configs/pidgin_baseline.yaml,')
print('then move this box to storage: !cp -r /content/artifacts /content/drive/MyDrive/')
print('and finally run the A/B benchmark (benchmarks/pidgin_eval/README.md).')

## After the run — non-negotiables

1. Copy `artifacts/` + new `experiments/*.json` to Google Drive **before** disconnecting.
2. `Runtime > Disconnect and delete runtime` (cost rule, PRD §11).
3. Push the config/experiment updates to GitHub — **never** weights or audio.
4. Run the A/B benchmark vs the untouched baseline per `benchmarks/pidgin_eval/README.md`
   — human listeners (≥3 Pidgin speakers) required before any 'sounds Pidgin' claim.